In [1]:
import os 
import torch


CLASSES_FILE = r'archive\class_names.txt'# r'data\archive\class_names.txt'
BATCH_SIZE=8 
IMG_SIZE=(640, 640)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

base_train = r'C:\Users\Sistem Cerdas Five\Desktop\vian\afre\archive\train'
train_gt_img = os.path.join(base_train, 'high')
train_low_img = os.path.join(base_train, 'low')
train_yolo_lbls = os.path.join(base_train, 'labels')

# base_val = '/kaggle/input/loli-augmented/val'
base_val = r'C:\Users\Sistem Cerdas Five\Desktop\vian\afre\archive\val'
val_gt_img = os.path.join(base_val, 'high')
val_low_img = os.path.join(base_val, 'low')
val_yolo_lbls = os.path.join(base_val, 'labels')

# base_test = '/kaggle/input/loli-augmented/test'
base_test = r'C:\Users\Sistem Cerdas Five\Desktop\vian\afre\archive\test'
test_gt_img = os.path.join(base_test, 'high')
test_low_img = os.path.join(base_test, 'low')
test_yolo_lbls = os.path.join(base_test, 'labels')


print(len(os.listdir(train_gt_img)), len(os.listdir(train_low_img)), len(os.listdir(train_yolo_lbls)))
print(len(os.listdir(val_gt_img)), len(os.listdir(val_low_img)), len(os.listdir(val_yolo_lbls)))
print(len(os.listdir(test_gt_img)), len(os.listdir(test_low_img)), len(os.listdir(test_yolo_lbls)))


6000 6000 6000
600 600 600
300 300 300


In [2]:
from modules.data_obj import LoLiStreetDataset_Direct 
from torch.utils.data import DataLoader
import torchvision.transforms as T


try:
    data_transform = T.ToTensor() 

    print("Initializing Train Dataset...")
    train_dataset = LoLiStreetDataset_Direct(
        high_dir=train_gt_img, 
        low_dir=train_low_img, 
        labels_dir=train_yolo_lbls, 
        img_size=(640, 640)
    )
    
    print("Initializing Val Dataset...")
    val_dataset = LoLiStreetDataset_Direct(
        high_dir=val_gt_img, 
        low_dir=val_low_img, 
        labels_dir=val_yolo_lbls, 
        img_size=(640, 640)
    )

    print("Initializing Test Dataset...")
    test_dataset = LoLiStreetDataset_Direct(
        high_dir=test_gt_img, 
        low_dir=test_low_img, 
        labels_dir=test_yolo_lbls, 
        img_size=(640, 640)
    )

    # Create DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=LoLiStreetDataset_Direct.collate_fn, num_workers=0, pin_memory=True)#, persistent_workers=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=LoLiStreetDataset_Direct.collate_fn, num_workers=0, pin_memory=True)#, persistent_workers=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=LoLiStreetDataset_Direct.collate_fn, num_workers=0, pin_memory=True)#, persistent_workers=True)

    print(f"\nSUCCESS: Datasets created. Train size: {len(train_dataset)}")

except Exception as e:
    print(f"\n[ERROR] Failed to initialize datasets: {e}")

Initializing Train Dataset...
Found 6000 valid triplets in C:\Users\Sistem Cerdas Five\Desktop\vian\afre\archive\train\high
Initializing Val Dataset...
Found 600 valid triplets in C:\Users\Sistem Cerdas Five\Desktop\vian\afre\archive\val\high
Initializing Test Dataset...
Found 300 valid triplets in C:\Users\Sistem Cerdas Five\Desktop\vian\afre\archive\test\high

SUCCESS: Datasets created. Train size: 6000


In [3]:
from models.illuminet import IllumiNet_b5_ARB


baseline = IllumiNet_b5_ARB(back_use_arb=False, dec_arb=False)
backARB = IllumiNet_b5_ARB(back_use_arb=True, dec_arb=False)
decARB = IllumiNet_b5_ARB(back_use_arb=False, dec_arb=True)
bothARB = IllumiNet_b5_ARB(back_use_arb=True, dec_arb=True)

355
Backbone ARB:  False
Decoder ARB False
Initializing nano params
NO ARB in Backbone
Night Dec initalized
Initializing nano params
Initializing nano params
✅ Loaded 162 layers into Backbone.
Missing keys: 0 | Unexpected keys: 0
✅ Loaded 108 layers into Neck.
Missing keys: 0 | Unexpected keys: 0
✅ Loaded 85 layers into Head.
Missing keys: 0 | Unexpected keys: 0
pretrain weights loaded
Backbone ARB:  True
Decoder ARB False
Initializing nano params
ARB Backbone Initialized
Night Dec initalized
Initializing nano params
Initializing nano params
✅ Loaded 162 layers into Backbone.
Missing keys: 72 | Unexpected keys: 0
✅ Loaded 108 layers into Neck.
Missing keys: 0 | Unexpected keys: 0
✅ Loaded 85 layers into Head.
Missing keys: 0 | Unexpected keys: 0
pretrain weights loaded
Backbone ARB:  False
Decoder ARB True
Initializing nano params
NO ARB in Backbone
ARB LLIE Decoder Initialized
Initializing nano params
Initializing nano params
✅ Loaded 162 layers into Backbone.
Missing keys: 0 | Unexpe

In [4]:
from training.train import ModelTrainer
from losses.yolo_loss import DetectionLoss
from losses.llie_loss import mean_abs_error


def_loss_gains = {'iou': 7.5, 'cls': 0.5, 'dfl': 1.5}
def_det_loss_fn = DetectionLoss(model=baseline, device=DEVICE, loss_gain=def_loss_gains)

baseline_trainer = ModelTrainer(
    model=baseline,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=1,
    det_loss_fn=def_det_loss_fn,
    enhance_loss_fn=mean_abs_error,
    experiment_name='baseline',
    device=DEVICE
)

baseline_trainer.fit()

c:\Users\Sistem Cerdas Five\Desktop\vian\afre\training\training\train.py:512: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()


⚠️ Warning: 'pyiqa' or 'lpips' not found. validation metrics will be skipped.
👉 Please run: pip install pyiqa lpips
Starting training on cuda for 1 epochs...


Epoch 0/1:   0%|          | 0/750 [00:00<?, ?it/s]c:\Users\Sistem Cerdas Five\Desktop\vian\afre\training\training\train.py:571: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
c:\Users\Sistem Cerdas Five\Desktop\vian\afre\training\losses\yolo_loss.py:428: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  anchor_points, stride_tensor = make_anchors(preds, torch.tensor(self.stride, device=self.device))
c:\Users\Sistem Cerdas Five\Desktop\vian\afre\venv\Lib\site-packages\torch\functional.py:534: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\TensorShape.cpp:3596.)
  return

RuntimeError: Index put requires the source and destination dtypes match, got Half for the destination and Float for the source.

In [ ]:

print('hello')

hello
